In [ ]:
import datasets
import json
from transformers import T5ForConditionalGeneration, T5Tokenizer
yue_zh_characters_validation = datasets.load_dataset("google/smol", "gatitos__yue_zh")
with open("character_classification.json", "r") as f:
    character_classfication = json.load(f)
    
yue_zh_characters_validation = datasets.load_dataset("google/smol", "gatitos__yue_zh")

characters = yue_zh_characters_validation["train"].to_pandas().sort_values(by="src", key=lambda x:x.str.len(), ascending=False)  # type: ignore

def swap_tokens(sentence: str) -> str:
    result = ""
    i = 0
    while i < len(sentence):
        found = False
        for k in range(len(characters)):
            if characters["src"][k] == sentence[i:i+len(characters["src"][k] )]:
                result = result + characters["trgs"][k][0].split(";")[0]
                i += len(characters["src"][k])
                found = True
                break
        if not found:
            result += sentence[i]
            i += 1
    
    return result

def swap_tokens_with_count(sentence: str) -> tuple[str, int]:
    result = ""
    swapped_count = 0
    i = 0
    while i < len(sentence):
        found = False
        for k in range(len(characters)):
            if characters["src"][k] == sentence[i:i+len(characters["src"][k] )]:
                result = result + characters["trgs"][k][0].split(";")[0]
                
                # only count the swaps that actually make a change
                if characters["trgs"][k][0].split(";")[0] != sentence[i:i+len(characters["src"][k])]: 
                    swapped_count += len(characters["trgs"][k][0].split(";")[0])
                    
                i += len(characters["src"][k])
                found = True
                break
        if not found:
            result += sentence[i]
            i += 1
    
    return result, swapped_count

# The first model is the publically available model, the second is our finetuned one (for 1 epoch of smolsent + smoldoc). 
# It is accessible for download here: https://drive.google.com/drive/folders/1I4NLBXJ1MvdmPWPlIUTZM_lDv81h5siK?usp=sharing
models = ['jbochi/madlad400-3b-mt', '/home/lasse-edslev/Downloads/checkpoint-784-20251215T145431Z-3-001/checkpoint-784']

with open('smolsent_data_split.json', 'r') as f:
    smolsent_data = json.load(f)
    
smolsent_test_data = smolsent_data['smol_test'] 
test_src, test_trg = smolsent_test_data['src'], smolsent_test_data['trg']

# It requires roughly 20 GB RAM to run this
def translate(model_name: str, srcs: list[str], trgs: list[str], do_token_swap: bool, write_to_file: str | None) -> list[dict]:
    model = T5ForConditionalGeneration.from_pretrained(model_name, device_map=None)
    tokenizer = T5Tokenizer.from_pretrained('jbochi/madlad400-3b-mt') # use the public tokenizer
    results = []
    for i, (src, trg) in enumerate(zip(srcs, trgs)):
        if do_token_swap:
            trg = swap_tokens(trg)
        
        input_ids = tokenizer('<2en> '+ trg, return_tensors="pt").input_ids
        out = model.generate(input_ids, max_length=256) # 256 to avoid cutting of
        result = tokenizer.decode(out[0], skip_special_tokens=True)
        results.append({
            'src': src,
            'trg': trg,
            'madlad_translation': result,
        })
        
        if i % 10 == 0:
            print(f'{i}/{len(test_src)}')
        
    if write_to_file is not None:
        with open(write_to_file, 'w+') as f:
            json.dump(results, f)
            
    return results
    
# Run the 4 combinations
for model_name in models:
    for do_token_swap in [True, False]:
        is_model_original = model_name == 'jbochi/madlad400-3b-mt'
        file_name = f'smolsent_test_{"tokenswap" if do_token_swap else ""}_translations_{"original" if is_model_original else "finetuned"}_madlad.json'
        translate(model_name=model_name, srcs=test_src, trgs=test_trg, do_token_swap=do_token_swap, write_to_file=file_name)